In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

def Poly_8(X):
    P_0 = X**0
    P_1 = X
    P_2 = X**2
    P_3 = X**3
    P_4 = X**4
    P_5 = X**5
    P_6 = X**6
    P_7 = X**7
    P_8 = X**8
    return np.column_stack((P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8))

def Ker(x):
    return 2 * np.sin(np.arcsin(2 * x - 1) / 3)

def LSM_ISD(S0, K, r, sd, T, I, M, alpha, alpha_opt, n):
    
    prices = []
    deltas = []
    gammas = []
    for i in range(n):
        dt = T/M
        M = max(2, int(M * T))
        df = np.exp(-r*dt)
        S = np.zeros((I, M + 1))
        U = np.random.uniform(0, 1, I)
        S_init = S0 + alpha*Ker(U)  
        S[:, 0] = S_init
        S[:, 1:] = (S_init)[:, np.newaxis] * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(I, M)), axis=1))

        deviations = S[:, 0] - S0
        mask = np.abs(deviations) <= alpha_opt
        count = np.sum(mask)

        if count > 0:
            U_new = np.random.uniform(0, 1, count)
            new_disp = alpha_opt * Ker(U_new)
            S_new0 = S0 + new_disp
            
            valid_mask = np.abs(S_new0 - S0) <= alpha_opt
            valid_count = np.sum(valid_mask)
        
            replace_indices = np.where(mask)[0][:valid_count]
            
            S[replace_indices, 0] = S_new0[valid_mask]
            S[replace_indices, 1:] = S_new0[valid_mask][:, np.newaxis] * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(valid_count, M)), axis=1))


        s = S/K 
        h = np.maximum(1 - s, 0)
        h[:, 0] = 0 
        h_copy = h.copy()

        for i in range(M, 1, -1):
            ITM = h[:, i - 1] > 0
            if np.sum(ITM) == 0:
                continue  

            Y = df * h_copy[ITM, i]
            X = s[ITM, i - 1]
            Z = Poly_8(X)

            reg1 = LinearRegression().fit(Z, Y)

            basis = Poly_8(s[:, i - 1])
            cont = reg1.predict(basis)

            h[:, i - 1] = np.where(h[:, i - 1] <= cont, 0, h[:, i - 1])
            h[h[:, i - 1] > 0, i:] = 0
            h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * df, h_copy[:, i - 1])

        df_vec = np.power(df, np.arange(M + 1)-1).reshape(-1, 1)

        disc_CashFlows = h @ df_vec

        Z_1 = disc_CashFlows.flatten()
        
        X_poly = Poly_8(s[:,1]) 
        
        
        reg2 = LinearRegression().fit(X_poly, Z_1)

        value_t1 = reg2.predict(X_poly)

        value = np.maximum(value_t1, np.maximum(1-s[:,1],0))* df * K



        
        outside_opt_mask = (S[:, 0] - S0 < -alpha_opt) | (S[:, 0] - S0 > alpha_opt)

        
        n_resim = np.sum(outside_opt_mask)


        if n_resim > 0:
            U_new = np.random.uniform(0, 1, n_resim)
            S_init_new = S0 + alpha_opt * Ker(U_new)
            
            S_new = np.zeros((n_resim, M + 1))
            S_new[:, 0] = S_init_new
            S_new[:, 1:] = (S_init_new)[:, np.newaxis] * np.exp(np.cumsum(
                (r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(n_resim, M)), axis=1))
            
            s_new = S_new / K
            h_new = np.maximum(1 - s_new, 0)
            h_new[:, 0] = 0
            h_copy_new = h_new.copy()

            for j in range(M, 1, -1):
                ITM = h_new[:, j - 1] > 0
                if np.sum(ITM) == 0:
                    continue
                Y = df * h_copy_new[ITM, j]
                X = s_new[ITM, j - 1]
                Z = Poly_8(X)
                reg1 = LinearRegression().fit(Z, Y)
                basis = Poly_8(s_new[:, j - 1])
                cont = reg1.predict(basis)
                h_new[:, j - 1] = np.where(h_new[:, j - 1] <= cont, 0, h_new[:, j - 1])
                h_new[h_new[:, j - 1] > 0, j:] = 0
                h_copy_new[:, j - 1] = np.where(h_new[:, j - 1] == 0, h_copy_new[:, j] * df, h_copy_new[:, j - 1])

            disc_CF_new = h_new @ df_vec
            Z_1_new = disc_CF_new.flatten()
            X_poly_new = Poly_8(s_new[:, 1])
            reg2_new = LinearRegression().fit(X_poly_new, Z_1_new)
            value_t1_new = reg2_new.predict(X_poly_new)
            value_new = np.maximum(value_t1_new, np.maximum(1 - s_new[:, 1], 0)) * df * K

            value[outside_opt_mask] = value_new
            S[outside_opt_mask, 0] = S_new[:, 0]

        X_poly2 = Poly_8(S[:, 0] - S0)
        reg3 = LinearRegression().fit(X_poly2, value)


        price = reg3.intercept_
        prices.append(price)

        delta = reg3.coef_[1]
        deltas.append(delta)

        gamma = 2* reg3.coef_[2]
        gammas.append(gamma)
    avg_price = np.mean(prices)    
    avg_delta = np.mean(deltas)
    avg_gamma = np.mean(gammas)

    price_std = np.std(prices, ddof=1)
    delta_std = np.std(deltas, ddof=1)
    gamma_std = np.std(gammas, ddof=1)




    return avg_price, price_std, avg_delta, avg_gamma, delta_std, gamma_std